# 04 - Model Comparison

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [ ]:
# Load cleaned data
df = pd.read_csv("cleaned.csv", dtype={"CloseMonth": str})
months = sorted(df["CloseMonth"].unique())
df.shape

## Train/test split

In [ ]:
# Test on the most recent month, train on the X months before it
X = 4
train = df[df["CloseMonth"].isin(months[-1 - X:-1])].copy()
test = df[df["CloseMonth"] == months[-1]].copy()
train.shape, test.shape

In [ ]:
# Normalize numeric features using the training set's mean and std
num_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt",
            "GarageSpaces", "ParkingTotal", "AssociationFee", "Latitude", "Longitude"]
mean, std = train[num_cols].mean(), train[num_cols].std()
train[num_cols] = (train[num_cols] - mean) / std
test[num_cols] = (test[num_cols] - mean) / std

## Models

In [ ]:
features = [c for c in df.columns if c not in ["ClosePrice", "CloseMonth"]]

models = {
    "LinearRegression (baseline)": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
}

rows = []
for name, model in models.items():
    model.fit(train[features], train["ClosePrice"])
    train_r2 = r2_score(train["ClosePrice"], model.predict(train[features]))
    test_r2 = r2_score(test["ClosePrice"], model.predict(test[features]))
    rows.append((name, train_r2, test_r2))

## Results

In [ ]:
results = pd.DataFrame(rows, columns=["model", "train_R2", "test_R2"])
results

## Model behavior

| Model | Train R² | Test R² |
|---|---|---|
| Linear Regression (baseline) | 0.58 | 0.60 |
| Decision Tree | 1.00 | 0.65 |
| Random Forest | 0.97 | 0.83 |

**Linear Regression**
- Strengths: fast, simple, and easy to interpret through its coefficients. Train and test R² are close, so it isn't overfitting.
- Weaknesses: it can only fit straight-line relationships, so it misses things like price per square foot changing by location. It is also sensitive to extreme prices.

**Decision Tree**
- Strengths: captures non-linear effects and interactions (like location × size) without any feature engineering. Easy to visualize.
- Weaknesses: with no depth limit it memorizes the training data (train R² ≈ 1.00) and generalizes poorly. Small changes in the data can produce a very different tree.

**Random Forest**
- Strengths: best test R² by a wide margin. Averaging many trees cancels out much of a single tree's overfitting while keeping the non-linear fit.
- Weaknesses: slower to train, harder to interpret than a single tree or linear model, and still overfits somewhat (train 0.97 vs test 0.83).
